# Lost in Handoff: clean baselines on a Colab A100

Runs the clean baselines (no faults, no added constraints) on Qwen3-8B through Ollama:
- the **single agent**
- the **depth-3 pipeline with free-text handoffs**

Each condition runs on 10 tasks, with 4 runs at a time on the GPU. Results go to Google Drive. Sweeps
resume, so rerunning a cell skips tasks that already have a result.

**Before you start:** Runtime → Change runtime type → **A100 GPU** → Save. Then Runtime → Run all.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Google Drive
Colab asks you to approve Drive access in a pop-up. Approve it yourself.
Put `lost-in-handoff-code.zip` anywhere in My Drive (ideally My Drive/lost-in-handoff/), or upload it in section 2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/lost-in-handoff'
OUT = f'{DRIVE_DIR}/runs/colab_a100/baseline'
!mkdir -p {OUT} && ls -la {DRIVE_DIR}

## 2. Code: unpack the bundle and fetch τ²-bench (same commit as local)
Looks for `lost-in-handoff-code*.zip` anywhere in My Drive (newest first, so a Drive copy renamed
`lost-in-handoff-code (1).zip` is found too). If none is found, an upload button appears: pick the zip
from your computer.

In [ ]:
import os, shutil, subprocess
ZIP = f'{DRIVE_DIR}/dist/lost-in-handoff-code.zip'
print('using', ZIP)
shutil.rmtree('/content/lost-in-handoff', ignore_errors=True)
subprocess.run(['unzip', '-q', ZIP, '-d', '/content/lost-in-handoff'], check=True)
assert os.path.exists('/content/lost-in-handoff/pyproject.toml'), 'zip has no pyproject.toml'
%cd /content/lost-in-handoff
!git clone -q https://github.com/sierra-research/tau2-bench vendor/tau2-bench
!git -C vendor/tau2-bench checkout -q 57e54143025bec219b5726884651c98eede202e2
!git -C vendor/tau2-bench rev-parse HEAD
!ls

## 3. Python dependencies (uv honors the local tau2 source in pyproject.toml; pip does not)

In [ ]:
!pip install -q uv
!uv pip install --system -q -e ".[dev]"
!python -m pytest -q 2>&1 | tail -1

## 4. Ollama server (4 runs at a time) and Qwen3-8B
`OLLAMA_NUM_PARALLEL=4` lets 4 requests share the loaded model. `OLLAMA_KEEP_ALIVE=-1` keeps it in GPU
memory between calls.

In [ ]:
!apt-get -qq install -y zstd > /dev/null   # the Ollama installer needs zstd; Colab lacks it
!curl -fsSL https://ollama.com/install.sh | sh
import os, subprocess, time, urllib.request
PARALLEL = 4
env = dict(os.environ, OLLAMA_NUM_PARALLEL=str(PARALLEL), OLLAMA_KEEP_ALIVE='-1')
subprocess.Popen('ollama serve > /content/ollama.log 2>&1', shell=True, env=env)
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434'); print('Ollama is up'); break
    except Exception:
        time.sleep(2)
else:
    !tail -20 /content/ollama.log
    raise RuntimeError('Ollama did not start; see the log above')
!ollama pull qwen3:8b
!ollama list

## 5. Quick check: one single-agent run of task 0
Should take a minute or two on an A100. The same run takes about 5 minutes on the M4 laptop.

In [ ]:
t0 = time.time()
!python -m handoff.cli run --task 0 --topology single --model qwen3:8b --out {DRIVE_DIR}/runs/colab_a100/smoke 2>&1 | grep -v DEBUG | tail -2
print(f'wall time: {time.time() - t0:.0f}s')

## 6. Baseline sweep: 4 shards at a time, resumable
Each shard takes every 4th task. Progress goes to `OUT/sweep.log`. If the session dies, reconnect, rerun
sections 1–4, then rerun this cell. Finished tasks are skipped.

In [ ]:
MODEL, LIMIT = 'qwen3:8b', 10
CONDITIONS = [('single', 3, 'free_text'), ('pipeline', 3, 'free_text')]
LOG = f'{OUT}/sweep.log'
for topo, depth, fmt in CONDITIONS:
    t0 = time.time()
    with open(LOG, 'a') as log:
        log.write(f'=== {topo} depth={depth} fmt={fmt} {time.strftime("%F %T")}\n')
    procs = [subprocess.Popen(
        f'python -m handoff.cli baseline --topology {topo} --depth {depth} --fmt {fmt} '
        f'--model {MODEL} --limit {LIMIT} --out {OUT} --shard {k}/{PARALLEL} 2>&1 '
        f'| grep -E --line-buffered "^task|Error|Traceback" >> {LOG}', shell=True)
        for k in range(1, PARALLEL + 1)]
    for p in procs:
        p.wait()
    print(f'{topo} {fmt}: done in {(time.time() - t0) / 60:.1f} min')
!tail -25 {LOG}

## 7. Summary

In [ ]:
import glob, json, collections
rows = collections.defaultdict(dict)
for f in glob.glob(f'{OUT}/*.json'):
    r = json.load(open(f))
    rows[r['condition']][r['task_id']] = r['success']
DO_NOTHING = {'10', '12'}   # these pass even if the agent does nothing (README decision 13)
for cond, res in sorted(rows.items()):
    meaningful = {t: ok for t, ok in res.items() if t not in DO_NOTHING}
    print(f'{cond:<26} {sum(res.values())}/{len(res)} solved   '
          f'({sum(meaningful.values())}/{len(meaningful)} without tasks 10 and 12)')
    print('   failed:', sorted((t for t, ok in res.items() if not ok), key=int))

## 8. Bring the results home
Download `My Drive/lost-in-handoff/runs/colab_a100/` and put it in the project's `runs/` folder, so the
analysis can use it alongside the laptop runs.